# Homework 3 - Pattern Recognition and Machine Learning

**k-Nearest Neighbors classification**

Two Gaussian classes in $\mathbb{R}^2$:

$$\omega_1 \sim \mathcal{N}\!\left(\begin{bmatrix}-1\\0\end{bmatrix},\,I\right),\qquad
  \omega_2 \sim \mathcal{N}\!\left(\begin{bmatrix}2\\2\end{bmatrix},\,I\right),$$

with $300$ samples per class.

kNN prediction at a query $x$: find the $k$ training points nearest to $x$ (under some metric), then take the majority class label.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

mu1 = np.array([-1.0, 0.0])
mu2 = np.array([ 2.0, 2.0])
Sigma = np.eye(2)

n = 300
X1 = np.random.multivariate_normal(mu1, Sigma, size=n)
X2 = np.random.multivariate_normal(mu2, Sigma, size=n)

X = np.vstack([X1, X2])
y = np.array([1] * n + [2] * n)

print("X shape:", X.shape)
print("Class 1 sample mean:", X1.mean(axis=0))
print("Class 2 sample mean:", X2.mean(axis=0))

# 1 / Implement kNN

## a) kNN classifier with Euclidean distance (from scratch)

For a query $x$:
1. compute distance to every training sample,
2. take the indices of the $k$ smallest distances,
3. return the majority class among them.

We implement a general metric, then specialise it to Euclidean / Manhattan / Chebyshev for task 3.

In [ ]:
def euclidean_distance(A, b):
    """Distances from rows of A (n, d) to a single point b (d,)."""
    return np.sqrt(np.sum((A - b) ** 2, axis=1))

def manhattan_distance(A, b):
    return np.sum(np.abs(A - b), axis=1)

def chebyshev_distance(A, b):
    return np.max(np.abs(A - b), axis=1)

def knn_predict_point(X_train, y_train, x_query, k=5, distance_fn=euclidean_distance):
    d = distance_fn(X_train, x_query)
    nn_idx = np.argpartition(d, k)[:k]
    counts = np.bincount(y_train[nn_idx])
    return np.argmax(counts)

def knn_predict(X_train, y_train, X_query, k=5, distance_fn=euclidean_distance):
    return np.array([
        knn_predict_point(X_train, y_train, x, k=k, distance_fn=distance_fn)
        for x in X_query
    ])

Helper for plotting decision regions on a 2D grid.

In [ ]:
def make_grid(X, pad=1.0, n_grid=200):
    x_min, x_max = X[:, 0].min() - pad, X[:, 0].max() + pad
    y_min, y_max = X[:, 1].min() - pad, X[:, 1].max() + pad
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, n_grid),
                         np.linspace(y_min, y_max, n_grid))
    grid = np.c_[xx.ravel(), yy.ravel()]
    return xx, yy, grid

def plot_decision_regions_knn(X_train, y_train, k, distance_fn, ax, title):
    xx, yy, grid = make_grid(X_train, pad=1.0, n_grid=180)
    Z = knn_predict(X_train, y_train, grid, k=k, distance_fn=distance_fn).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, levels=[0.5, 1.5, 2.5])
    ax.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], s=12, alpha=0.7, label=r"$\omega_1$")
    ax.scatter(X_train[y_train == 2, 0], X_train[y_train == 2, 1], s=12, alpha=0.7, label=r"$\omega_2$")
    ax.set_title(title); ax.set_xlabel("x1"); ax.set_ylabel("x2")
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend()

## b) Visualise decision regions for $k = 1, 5, 15$

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, k in zip(axes, [1, 5, 15]):
    plot_decision_regions_knn(X, y, k=k, distance_fn=euclidean_distance,
                              ax=ax, title=f'k-NN, Euclidean, k = {k}')
plt.tight_layout()
plt.show()

## c) Discussion — how does the decision boundary change with $k$?

- **$k = 1$.** Each training point owns its own Voronoi cell, so the boundary is highly *irregular* and the classifier perfectly memorises the training labels. Low bias, very high variance — even a single noisy point near the boundary creates a small island of the wrong class.
- **$k = 5$.** Majority over 5 neighbours averages out individual noise points; the boundary becomes noticeably smoother but still bends to follow the local shape of the data.
- **$k = 15$.** The boundary is close to the optimal linear separator implied by the two equal-covariance Gaussians. Variance is small, but the boundary now ignores small-scale structure (bias has grown).

Larger $k$ ↔ more averaging ↔ smoother boundary ↔ higher bias, lower variance. In the limit $k = n$, every query gets the global majority class — the classifier becomes constant.

# 2 / Bias–variance behaviour

## a) Stratified 70/30 train/test split

In [ ]:
def stratified_split(X, y, test_size=0.3, seed=1):
    rng = np.random.default_rng(seed)
    idx_tr, idx_te = [], []
    for c in np.unique(y):
        idx_c = np.where(y == c)[0]
        rng.shuffle(idx_c)
        n_test = int(round(len(idx_c) * test_size))
        idx_te.append(idx_c[:n_test])
        idx_tr.append(idx_c[n_test:])
    idx_tr = np.concatenate(idx_tr); rng.shuffle(idx_tr)
    idx_te = np.concatenate(idx_te); rng.shuffle(idx_te)
    return X[idx_tr], X[idx_te], y[idx_tr], y[idx_te]

X_train, X_test, y_train, y_test = stratified_split(X, y, test_size=0.3, seed=1)
print("Train:", X_train.shape, "  Test:", X_test.shape)
print("Class counts (train):", np.bincount(y_train)[1:])
print("Class counts (test):",  np.bincount(y_test)[1:])

## b–c) Evaluate kNN for $k = 1, \dots, 50$ and plot test accuracy

In [ ]:
k_values = np.arange(1, 51)
test_accs = []
train_accs = []

for k in k_values:
    y_pred_te = knn_predict(X_train, y_train, X_test,  k=k)
    y_pred_tr = knn_predict(X_train, y_train, X_train, k=k)
    test_accs.append((y_pred_te == y_test).mean())
    train_accs.append((y_pred_tr == y_train).mean())

test_accs = np.array(test_accs)
train_accs = np.array(train_accs)

best_k = k_values[np.argmax(test_accs)]
print(f"Best k on this split: {best_k}    test acc = {test_accs.max():.4f}")

plt.figure(figsize=(8, 4.5))
plt.plot(k_values, train_accs, marker='.', label='train accuracy')
plt.plot(k_values, test_accs,  marker='o', label='test accuracy', markersize=4)
plt.axvline(best_k, color='red', linestyle='--', alpha=0.5, label=f'best k = {best_k}')
plt.xlabel('k'); plt.ylabel('accuracy')
plt.title('kNN train/test accuracy as a function of k')
plt.grid(True, alpha=0.3); plt.legend()
plt.show()

### Discussion — why accuracy drops at both ends

- **Very small $k$ (e.g. $k = 1$): high variance.**
  The prediction depends on a *single* nearby training point. Random noise in the position of that point — or a single mislabelled / overlapping sample — flips the prediction. The training accuracy is $\approx 1$ (the model memorises its own data), but test accuracy fluctuates and is generally lower.
- **Very large $k$ (e.g. $k \to n_\mathrm{train}$): high bias.**
  The majority is taken over a large neighbourhood that no longer reflects local class structure. As $k$ grows past a critical size, the neighbourhood always contains many points of *both* classes, and the prediction tends towards the globally more frequent class — accuracy drops to the prior of the majority class (here $\approx 0.5$).
- **Sweet spot in between.**
  Test accuracy peaks at an intermediate $k$ that balances bias and variance. For these well-separated, equal-covariance Gaussians, the optimum is around $k = 10\,\text{–}\,25$. This is exactly the U-shape (inverted U for accuracy) predicted by the bias–variance decomposition.

# 3 / Distance metrics

Compare Euclidean, Manhattan and Chebyshev distance at a fixed $k$.

$$\|a-b\|_2 = \sqrt{\textstyle\sum_i (a_i-b_i)^2},\quad
  \|a-b\|_1 = \textstyle\sum_i |a_i-b_i|,\quad
  \|a-b\|_\infty = \max_i |a_i-b_i|.$$

In [ ]:
k_fixed = 11
metrics = [
    ("Euclidean", euclidean_distance),
    ("Manhattan", manhattan_distance),
    ("Chebyshev", chebyshev_distance),
]

accs = {}
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (name, dist_fn) in zip(axes, metrics):
    y_pred = knn_predict(X_train, y_train, X_test, k=k_fixed, distance_fn=dist_fn)
    acc = (y_pred == y_test).mean()
    accs[name] = acc
    plot_decision_regions_knn(X_train, y_train, k=k_fixed, distance_fn=dist_fn,
                              ax=ax, title=f'{name},  k = {k_fixed},  acc = {acc:.3f}')
plt.tight_layout()
plt.show()

print("Test accuracies:")
for name, a in accs.items():
    print(f"  {name:>10}: {a:.4f}")

### Discussion — why metrics matter

Different metrics define different *neighbourhoods*:

- Euclidean → circular neighbourhoods, isotropic — no preferred direction.
- Manhattan → diamond-shaped neighbourhoods rotated 45°; axes are privileged. Decision regions tend to have axis-aligned facets.
- Chebyshev → square neighbourhoods aligned with the axes; one large coordinate difference dominates.

For these isotropic Gaussians with $\Sigma = I$, all three metrics work essentially equally well (accuracies within ~1–2 %). Visually the boundaries differ in their *shape*: Manhattan and Chebyshev produce slightly more axis-aligned, blockier regions; Euclidean is smoothest.

**When does the metric matter?**
When features have *different scales*, or when correlations / anisotropic noise mean that some directions are intrinsically more informative than others. Euclidean implicitly treats all coordinates symmetrically, which is only correct after appropriate scaling (or a whitening transformation). Manhattan can be more robust to outliers in individual coordinates; Chebyshev is very sensitive to whichever feature has the largest spread. A practical first step before using kNN is therefore to **standardise the features**, after which the choice of $\ell_p$ norm matters far less.

# 4 / Curse of dimensionality

Same mean separation in every coordinate ($\mu_1 = \mathbf 0$, $\mu_2 = \mathbf 1$), but the dimension grows. We measure how kNN accuracy degrades with $d$.

In [ ]:
def generate_dataset(d, n_per_class=400, seed=0):
    rng = np.random.default_rng(seed)
    X1 = rng.standard_normal((n_per_class, d)) + np.zeros(d)
    X2 = rng.standard_normal((n_per_class, d)) + np.ones(d)
    X = np.vstack([X1, X2])
    y = np.array([1] * n_per_class + [2] * n_per_class)
    return X, y

dims = [2, 5, 10, 20, 50, 100]
k_fixed = 11
acc_per_dim = []

for d in dims:
    X_d, y_d = generate_dataset(d, n_per_class=400, seed=d)
    X_tr, X_te, y_tr, y_te = stratified_split(X_d, y_d, test_size=0.3, seed=d)
    y_pred = knn_predict(X_tr, y_tr, X_te, k=k_fixed)
    acc = (y_pred == y_te).mean()
    acc_per_dim.append(acc)
    print(f"d = {d:>3}    test accuracy = {acc:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(dims, acc_per_dim, marker='o')
plt.xscale('log')
plt.xlabel('dimension d (log scale)')
plt.ylabel('test accuracy')
plt.title(f'kNN accuracy vs. dimension  (k = {k_fixed})')
plt.grid(True, alpha=0.3, which='both')
plt.show()

Note an apparent paradox: with $\mu_2 - \mu_1 = \mathbf 1$, the Bayes-optimal error actually *decreases* with $d$ (the mean separation grows as $\sqrt d$). So why does kNN often look worse in high $d$? Because distances themselves *concentrate*: nearest and farthest points become almost equidistant, so the notion of "nearest neighbour" loses meaning. Let's measure that directly.

In [ ]:
def nearest_farthest_ratio(d, n_points=500, seed=0):
    rng = np.random.default_rng(seed)
    pts = rng.standard_normal((n_points, d))
    q = rng.standard_normal(d)
    dists = np.sqrt(np.sum((pts - q) ** 2, axis=1))
    return dists.min() / dists.max()

dims_full = [1, 2, 5, 10, 20, 50, 100, 200]
ratios = [np.mean([nearest_farthest_ratio(d, seed=s) for s in range(50)]) for d in dims_full]

plt.figure(figsize=(7, 4))
plt.plot(dims_full, ratios, marker='x')
plt.xscale('log')
plt.xlabel('dimension d (log scale)')
plt.ylabel('min / max distance ratio')
plt.title('Distance concentration: nearest/farthest ratio')
plt.grid(True, alpha=0.3, which='both')
plt.show()

### Discussion — why kNN typically degrades in high $d$

- **Distance concentration.**
  For random points in high dimension, the ratio $\min\!\text{dist} / \max\!\text{dist}$ approaches 1. All points look roughly equidistant to a query, so picking the "nearest" $k$ is dominated by noise rather than by genuine proximity.
- **Volume / sparsity.**
  The volume of the unit ball grows like $r^d$; to contain a constant fraction of the data, the radius of a $k$-NN ball must grow with $d$. Effectively, neighbours are no longer *local*.
- **Sample requirement.**
  To keep the local approximation accurate, $n$ has to scale roughly exponentially with $d$. With fixed $n$, classification gradually becomes guessing.
- **Feature relevance.**
  In high $d$ many features are usually noise. Each adds to the distance equally under standard metrics, drowning out the few informative ones.

In our experiment, the test accuracy *does not* collapse to 0.5 because the mean separation also grows with $d$ — but the *gap* to the Bayes-optimal rate widens. In real high-dimensional problems with a fixed signal, kNN typically deteriorates quickly, which is why dimensionality reduction, feature selection, or metric learning are commonly applied before using kNN.

-----
## How do they do it in practice — scikit-learn

Cross-check our from-scratch implementation against `sklearn.neighbors.KNeighborsClassifier`.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

print("=== Task 2 (k = best from our sweep) ===")
for k in [1, 5, 15, int(best_k)]:
    clf = KNeighborsClassifier(n_neighbors=k, metric='minkowski', p=2)
    clf.fit(X_train, y_train)
    print(f"  k = {k:>3}   sklearn test acc = {clf.score(X_test, y_test):.4f}")

print("\n=== Task 3 (k = 11, different metrics) ===")
for name, sk_metric in [("Euclidean", "euclidean"),
                        ("Manhattan", "manhattan"),
                        ("Chebyshev", "chebyshev")]:
    clf = KNeighborsClassifier(n_neighbors=11, metric=sk_metric)
    clf.fit(X_train, y_train)
    print(f"  {name:>10}: sklearn test acc = {clf.score(X_test, y_test):.4f}")

print("\n=== Task 4 (curse of dimensionality, k = 11) ===")
for d in dims:
    X_d, y_d = generate_dataset(d, n_per_class=400, seed=d)
    X_tr, X_te, y_tr, y_te = stratified_split(X_d, y_d, test_size=0.3, seed=d)
    clf = KNeighborsClassifier(n_neighbors=11).fit(X_tr, y_tr)
    print(f"  d = {d:>3}    sklearn test acc = {clf.score(X_te, y_te):.4f}")